# 📊 Caption quality: evaluation & experimentation

The point of this notebook is **measurement**, not just generation. We:
1. Set up BLIP + a small eval set.
2. Measure a baseline (CLIPScore + BLEU).
3. Experiment with decoding strategies (greedy / beam / sampling) and see what moves.
4. LoRA fine-tune BLIP and measure before vs after.
5. Summarise findings honestly — including null results.

Runs on a **free Colab T4**. Set *Runtime → Change runtime type → T4 GPU*.

> Resume framing: this demonstrates you can *evaluate* models rigorously, not just call them. A clean null result, well measured, is a strong signal.

## 1. Install & load

In [15]:
!pip install -q transformers peft datasets accelerate pillow
!pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 54.1 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [2]:
import torch
from transformers import BlipForConditionalGeneration, BlipProcessor
from datasets import load_dataset

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_id = 'Salesforce/blip-image-captioning-base'
processor = BlipProcessor.from_pretrained(model_id)
model = BlipForConditionalGeneration.from_pretrained(model_id).to(device).eval()
print('loaded on', device)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identic

loaded on cuda


In [6]:
eval_ds = load_dataset('Mozilla/flickr30k-transformed-captions', split='test[:100]')
print(eval_ds)
print('example refs:', eval_ds[0]['original_alt_text'][:2])

README.md: 0.00B [00:00, ?B/s]

data/test-00000-of-00009.parquet:   0%|          | 0.00/459M [00:00<?, ?B/s]

data/test-00001-of-00009.parquet:   0%|          | 0.00/463M [00:00<?, ?B/s]

data/test-00002-of-00009.parquet:   0%|          | 0.00/474M [00:00<?, ?B/s]

data/test-00003-of-00009.parquet:   0%|          | 0.00/461M [00:00<?, ?B/s]

...

data/test-00006-of-00009.parquet:   0%|          | 0.00/518M [00:00<?, ?B/s]

data/test-00007-of-00009.parquet:   0%|          | 0.00/497M [00:00<?, ?B/s]

data/test-00008-of-00009.parquet:   0%|          | 0.00/466M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/31014 [00:00<?, ? examples/s]

Dataset({
    features: ['image', 'alt_text', 'sentids', 'split', 'img_id', 'filename', 'original_alt_text'],
    num_rows: 100
})
example refs: ['Two young guys with shaggy hair look at their hands while hanging out in the yard.', 'Two young, White males are outside near many bushes.']


## 2. Eval set
A small slice of Flickr30k. Each image has several human reference captions — we need those for BLEU.

## 3. Helper: generate + the two metrics
CLIPScore is reference-free (image vs caption). BLEU is reference-based (caption vs human captions).

In [7]:
from PIL import Image
from transformers import CLIPModel, CLIPProcessor

clip_id = 'openai/clip-vit-base-patch32'
clip_proc = CLIPProcessor.from_pretrained(clip_id)
clip_model = CLIPModel.from_pretrained(clip_id).to(device).eval()

@torch.no_grad()
def generate(images, **gen_kwargs):
    caps = []
    for img in images:
        inp = processor(img.convert('RGB'), return_tensors='pt').to(device)
        out = model.generate(**inp, **gen_kwargs)
        caps.append(processor.decode(out[0], skip_special_tokens=True).strip())
    return caps

@torch.no_grad()
def clip_score(images, captions):
    sims = []
    for img, cap in zip(images, captions):
        inp = clip_proc(text=[cap], images=img.convert('RGB'),
                        return_tensors='pt', padding=True, truncation=True).to(device)
        o = clip_model(**inp)
        ie = o.image_embeds / o.image_embeds.norm(dim=-1, keepdim=True)
        te = o.text_embeds / o.text_embeds.norm(dim=-1, keepdim=True)
        sims.append(float((ie @ te.T).item()))
    return round(sum(sims)/len(sims)*100, 2)


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
# Self-contained BLEU (no repo needed) — same logic as src/captioner/evaluation.py
import math
from collections import Counter

def _tokenize(text):
    return text.lower().split()

def bleu_score(candidates, references, max_n=4):
    """Corpus BLEU (0-100). references[i] = list of human captions for candidate i."""
    assert len(candidates) == len(references)
    shortest = min((len(_tokenize(c)) for c in candidates), default=1)
    eff_n = max(1, min(max_n, shortest))
    clipped = [0]*eff_n; totals = [0]*eff_n; cand_len = ref_len = 0
    for cand, refs in zip(candidates, references):
        ctoks = _tokenize(cand); cand_len += len(ctoks)
        rlens = [len(_tokenize(r)) for r in refs]
        ref_len += min(rlens, key=lambda rl: (abs(rl-len(ctoks)), rl))
        for n in range(1, eff_n+1):
            cgrams = Counter(tuple(ctoks[i:i+n]) for i in range(len(ctoks)-n+1))
            maxref = Counter()
            for r in refs:
                rtoks = _tokenize(r)
                rgrams = Counter(tuple(rtoks[i:i+n]) for i in range(len(rtoks)-n+1))
                for g, c in rgrams.items():
                    maxref[g] = max(maxref[g], c)
            clipped[n-1] += sum(min(c, maxref[g]) for g, c in cgrams.items())
            totals[n-1] += max(sum(cgrams.values()), 1)
    precisions = [(clipped[i]/totals[i]) if totals[i] else 0.0 for i in range(eff_n)]
    geo = math.exp(sum(math.log(p) for p in precisions)/eff_n) if min(precisions) > 0 else 0.0
    bp = 1.0 if cand_len > ref_len else math.exp(1 - ref_len/max(cand_len, 1))
    return round(bp*geo*100, 2)

print('BLEU ready. self-test (identical):', bleu_score(['a dog runs in the park'], [['a dog runs in the park']]))


BLEU ready. self-test (identical): 100.0


## 4. Baseline measurement
Default (greedy) decoding. This is the number everything else is compared to.

In [10]:
images = [eval_ds[i]['image'] for i in range(len(eval_ds))]
refs   = [eval_ds[i]['original_alt_text'] for i in range(len(eval_ds))] # Changed 'caption' to 'original_alt_text'

base_caps = generate(images, max_new_tokens=40)
base_clip = clip_score(images, base_caps)
base_bleu = bleu_score(base_caps, refs)
print(f'BASELINE  CLIPScore={base_clip}  BLEU={base_bleu}')
for i in range(3): print(' -', base_caps[i])

BASELINE  CLIPScore=28.16  BLEU=29.54
 - a man standing in the grass
 - a metal tower
 - a little girl in a pink dress


## 5. Experiment: decoding strategies
Same model, no training — just change how we sample tokens. We sweep a few settings and record both metrics for each.

In [11]:
experiments = {
    'greedy':        dict(max_new_tokens=40),
    'beam_5':        dict(max_new_tokens=40, num_beams=5),
    'sample_t0.7':   dict(max_new_tokens=40, do_sample=True, temperature=0.7),
    'sample_topp0.9':dict(max_new_tokens=40, do_sample=True, top_p=0.9),
    'beam_5_long':   dict(max_new_tokens=60, num_beams=5, min_new_tokens=20),
}

results = {}
for name, kw in experiments.items():
    caps = generate(images, **kw)
    results[name] = {'clip': clip_score(images, caps),
                     'bleu': bleu_score(caps, refs),
                     'example': caps[0]}
    print(f"{name:16s} CLIP={results[name]['clip']:5.2f}  BLEU={results[name]['bleu']:5.2f}")


greedy           CLIP=28.16  BLEU=29.54
beam_5           CLIP=28.25  BLEU=23.30
sample_t0.7      CLIP=26.42  BLEU=20.32
sample_topp0.9   CLIP=26.20  BLEU=13.30
beam_5_long      CLIP=31.30  BLEU=15.76


In [12]:
import pandas as pd
df = pd.DataFrame(results).T[['clip','bleu','example']]
df


,clip,bleu,example
greedy,28.16,29.54,a man standing in the grass
beam_5,28.25,23.3,the man is wearing a blue shirt
sample_t0.7,26.42,20.32,the man is outside
sample_topp0.9,26.2,13.3,the tree near the man
beam_5_long,31.3,15.76,a man in a blue shirt is standing in front of ...


## 6. LoRA fine-tune
Train only small adapter matrices. We push toward *more detailed* captions so there's a measurable change to evaluate.

In [16]:
from peft import LoraConfig, get_peft_model

train_ds = load_dataset('Mozilla/flickr30k-transformed-captions', split='test[100:500]')
lora = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05,
                  target_modules=['query','value'])
model = get_peft_model(model, lora)
model.print_trainable_parameters()


trainable params: 589,824 || all params: 248,034,424 || trainable%: 0.2378


In [18]:
from torch.utils.data import DataLoader

model.train()
opt = torch.optim.AdamW(model.parameters(), lr=5e-4);

def collate(batch):
    imgs = [ex['image'].convert('RGB') for ex in batch]
    caps = [max(ex['original_alt_text'], key=len) for ex in batch]  # changed 'caption' to 'original_alt_text'
    enc = processor(images=imgs, text=caps, padding=True, return_tensors='pt')
    enc['labels'] = enc['input_ids'].clone()
    return {k: v.to(device) for k, v in enc.items()}

loader = DataLoader(train_ds, batch_size=4, shuffle=True, collate_fn=collate)
for epoch in range(2):
    for i, batch in enumerate(loader):
        opt.zero_grad(); out = model(**batch); out.loss.backward(); opt.step()
        if i % 25 == 0: print(f'epoch {epoch} step {i} loss {out.loss.item():.3f}')
model.eval()

epoch 0 step 0 loss 4.416
epoch 0 step 25 loss 3.816
epoch 0 step 50 loss 4.994
epoch 0 step 75 loss 4.693
epoch 1 step 0 loss 4.396
epoch 1 step 25 loss 5.588
epoch 1 step 50 loss 3.385
epoch 1 step 75 loss 4.547


PeftModel(
  (base_model): LoraModel(
    (model): BlipForConditionalGeneration(
      (vision_model): BlipVisionModel(
        (embeddings): BlipVisionEmbeddings(
          (patch_embedding): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
        )
        (encoder): BlipEncoder(
          (layers): ModuleList(
            (0-11): 12 x BlipEncoderLayer(
              (self_attn): BlipAttention(
                (dropout): Dropout(p=0.0, inplace=False)
                (qkv): Linear(in_features=768, out_features=2304, bias=True)
                (projection): Linear(in_features=768, out_features=768, bias=True)
              )
              (layer_norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
              (mlp): BlipMLP(
                (activation_fn): GELUActivation()
                (fc1): Linear(in_features=768, out_features=3072, bias=True)
                (fc2): Linear(in_features=3072, out_features=768, bias=True)
              )
              (layer_norm2):

## 7. Before vs after
The headline comparison. Measure the fine-tuned model on the SAME eval set.

In [19]:
ft_caps = generate(images, max_new_tokens=40)
ft_clip = clip_score(images, ft_caps)
ft_bleu = bleu_score(ft_caps, refs)

print(f'BEFORE  CLIP={base_clip}  BLEU={base_bleu}')
print(f'AFTER   CLIP={ft_clip}  BLEU={ft_bleu}')
print(f'DELTA   CLIP={ft_clip-base_clip:+.2f}  BLEU={ft_bleu-base_bleu:+.2f}')
for i in range(3):
    print(f'\nimg {i}:\n  before: {base_caps[i]}\n  after : {ft_caps[i]}')


BEFORE  CLIP=28.16  BLEU=29.54
AFTER   CLIP=33.25  BLEU=13.94
DELTA   CLIP=+5.09  BLEU=-15.60

img 0:
  before: a man standing in the grass
  after : a man in a blue shirt and jeans is standing in a garden with a green bush behind him and a white fence behind him

img 1:
  before: a metal tower
  after : a man in a black jacket is on a high voltage power line with a helmet on his head and a helmet on his head is on a metal structure that is attached to a pole that is holding a

img 2:
  before: a little girl in a pink dress
  after : a little girl in a pink dress is standing on a wooden platform with a chicken coop in the background and a blue bucket in the foreground


## 8. Save adapter & write up findings


In [20]:
model.save_pretrained('blip-lora-adapter')
print('saved adapter (a few MB)')


saved adapter (a few MB)


### Interpreting your results — write this honestly in the README

- If CLIPScore barely moved: expected. BLIP is already strong on general captions. Report the delta and explain *why* it's small.
- If BLEU rose but CLIPScore fell (or vice versa): a great talking point — the metrics measure different things (n-gram overlap vs semantic match).
- Which decoding strategy won, and the quality/length trade-off you observed.

The deliverable is the **table and the interpretation**, not a big number.